In [67]:
import json
import pandas as pd

with open("../../dataset/train_sentiment.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(df.head())
print(df["sentiment"].value_counts())

                                                text sentiment
0  มาตามนัด 🚮🗑️ . . #เขตพญาไท #กรุงเทพมหานคร @สาย...  positive
1  &#9888; แยกราชเทวีจะรื้อน้ำพุกี่โมง? &#128591;...  negative
2  &#128308;สด!"ไอซ์ รักชนก" แถลงข่าวการประมูลคลื...  negative
3  &#127808;#เขตหนองจอก >>ติดตามการดำเนินงานโครงก...   neutral
4  ไทวัสดุ ส่งช่างมือ 1 วีฟิกซ์ ร่วมฟื้นฟู ปรับภู...  positive
sentiment
positive    3000
negative    3000
neutral     3000
Name: count, dtype: int64


In [ ]:
import re, html

def clean_text(text):
    text = html.unescape(text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text.strip()

df["text"] = df["text"].apply(clean_text)

text = df["text"]
sentiment = df["sentiment"]

In [69]:
from sklearn.model_selection import train_test_split

text_train, text_val, sentiment_train, sentiment_val = train_test_split(
    text, sentiment, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from pythainlp.tokenize import word_tokenize

def tokenize(text):
    return word_tokenize(text, engine="newmm")

vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    ngram_range=(1,2),
    max_df=0.85,
    min_df=3
)

text_train_tfidf = vectorizer.fit_transform(text_train)
text_val_tfidf = vectorizer.transform(text_val)

c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score


model = LinearSVC()
model.fit(text_train_tfidf, sentiment_train)

y_pred = model.predict(text_val_tfidf)
print(classification_report(sentiment_val, y_pred))
print("Accuracy:", accuracy_score(sentiment_val, y_pred))

              precision    recall  f1-score   support

    negative       0.83      0.87      0.85       611
     neutral       0.77      0.68      0.73       617
    positive       0.79      0.85      0.82       572

    accuracy                           0.80      1800
   macro avg       0.80      0.80      0.80      1800
weighted avg       0.80      0.80      0.80      1800

Accuracy: 0.7977777777777778


In [ ]:
import pickle

with open("sentiment_model_new.pkl", "wb") as f:
    pickle.dump((vectorizer, model), f)